# 03 · CS Trivia for Squarepoint (with C++)
Round 1 has ~20 minutes of quick-fire questions: Python, OS, networking, Linux, C++, plus a little SQL/git/ML.

**How to use it**
- Every question has a collapsed answer. **Say your answer out loud first**, then open it.
- ⭐ = reported for Squarepoint or very likely. Do all ⭐ first, then the rest.
- Answers are written in **interview length** (30–60 seconds spoken). "Go deeper" lines are for follow-ups.
- Code cells are runnable demos. C++ demos compile with `g++`/`clang++` if you have one (on a Mac: `xcode-select --install`).

**Answer formula (use it for every question):** *one-line definition → how it works → tradeoff or example → (bonus) why it matters in trading.*

| Section | ⭐ count | Priority |
|---|---|---|
| 1 Python internals | many | 🔴 highest (reported) |
| 2 Concurrency & async | many | 🔴 |
| 3 OS: processes, memory, real-time | many | 🔴 (reported: virtual memory, page tables, RT systems) |
| 4 Networking | some | 🟠 (reported: TCP vs UDP) |
| 5 Linux / Unix | some | 🟠 (reported) |
| 6 C++ | many | 🔴 (it's on your résumé, and smart pointers were reported) |
| 7 Git · 8 SQL · 9 DS · 10 ML | few | 🟡 skim |
| 11 Finance context · 12 Your projects · 13 Questions to ask | | 🟠 |

In [1]:
# Helper for the C++ demos. Needs g++ or clang++ on your PATH.
import shutil, subprocess, tempfile, os, textwrap

CXX = shutil.which("g++") or shutil.which("clang++")

def run_cpp(src, flags="-std=c++20 -O0 -Wall"):
    if not CXX:
        print("No C++ compiler found. Read the code; the expected output is in the comments.")
        return
    d = tempfile.mkdtemp()
    cpp, exe = os.path.join(d, "a.cpp"), os.path.join(d, "a.out")
    open(cpp, "w").write(textwrap.dedent(src))
    r = subprocess.run(f"{CXX} {flags} {cpp} -o {exe}", shell=True, capture_output=True, text=True)
    if r.returncode:
        print("COMPILE ERROR\n", r.stderr); return
    if r.stderr.strip():
        print("[warnings]\n", r.stderr)
    out = subprocess.run(exe, capture_output=True, text=True)
    print(out.stdout + out.stderr)

print("compiler:", CXX)

compiler: /usr/bin/g++


---
# 1 · Python internals
Reported for Squarepoint: generators, GC, context managers, tuples vs lists, hashing, GIL, decorators, `yield`, "a key problem with Python in finance".

**Q1. ⭐ What is CPython, and how does Python code run?**

<details><summary>Answer</summary>

CPython is the reference interpreter, written in C. Your `.py` source is **compiled to bytecode** (`.pyc`, cached in `__pycache__`), and a **stack-based virtual machine** interprets that bytecode one instruction at a time. There is no ahead-of-time compilation to machine code, which is one reason Python is slower than C++.
*Go deeper:* `dis.dis(fn)` shows the bytecode. PyPy is an alternative interpreter with a JIT. Python 3.13 added an experimental JIT and an optional free-threaded (no-GIL) build.
</details>

**Q2. ⭐ What is the GIL (Global Interpreter Lock)? Why does it exist?**

<details><summary>Answer</summary>

The GIL is a mutex in CPython that allows **only one thread to execute Python bytecode at a time** within a process.
It exists because CPython's memory management (**reference counting**) isn't thread-safe. One global lock is simpler and faster for single-threaded code than fine-grained locks on every object.
**Consequence:** threads **don't speed up CPU-bound** Python code. They still help with **I/O-bound** work, because the GIL is released while waiting on I/O (sockets, files, `time.sleep`) and inside many C extensions (NumPy releases it during heavy computation).
*Go deeper:* PEP 703 makes the GIL optional (free-threaded build `python3.13t`, still experimental).
</details>

**Q3. ⭐ Multithreading vs multiprocessing in Python: when do you use which?**

<details><summary>Answer</summary>

| | `threading` | `multiprocessing` |
|---|---|---|
| Memory | shared address space | separate processes, separate memory |
| GIL | one GIL shared: **no CPU parallelism** | one GIL per process: **true parallelism** |
| Good for | I/O-bound work (network calls, disk, DB) | CPU-bound work (number crunching, parsing) |
| Cost | cheap to create; need locks for shared state | expensive to start; data must be **pickled** to move between processes (IPC) |

Rule of thumb: **I/O-bound → threads or asyncio; CPU-bound → processes (or NumPy/C extensions that release the GIL).** `concurrent.futures` gives both the same API: `ThreadPoolExecutor` / `ProcessPoolExecutor`.
</details>

In [ ]:
# Demo: CPU-bound work: threads don't help, processes do (numbers vary by machine)
import time, threading
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def burn(n=3_000_000):
    s = 0
    for i in range(n):
        s += i * i
    return s

t0 = time.perf_counter(); [burn() for _ in range(4)]; seq = time.perf_counter() - t0
t0 = time.perf_counter()
with ThreadPoolExecutor(4) as ex: list(ex.map(burn, [3_000_000] * 4))
thr = time.perf_counter() - t0
print(f"sequential {seq:.2f}s | 4 threads {thr:.2f}s  (≈ same: the GIL)")

# Processes: in a notebook on macOS/Windows this may need the function in a .py file.
try:
    t0 = time.perf_counter()
    with ProcessPoolExecutor(4) as ex: list(ex.map(burn, [3_000_000] * 4))
    print(f"4 processes {time.perf_counter() - t0:.2f}s  (faster if you have ≥4 cores)")
except Exception as e:
    print("process pool skipped in this environment:", type(e).__name__)

# I/O-bound: threads DO help, because sleep/network releases the GIL
def io_task(_): time.sleep(0.5)
t0 = time.perf_counter()
with ThreadPoolExecutor(4) as ex: list(ex.map(io_task, range(4)))
print(f"4 × 0.5s I/O waits with threads: {time.perf_counter() - t0:.2f}s (≈0.5s, not 2s)")

**Q4. ⭐ How does Python manage memory? What does the garbage collector do?**

<details><summary>Answer</summary>

Two mechanisms:
1. **Reference counting (primary).** Every object stores a count of references to it. When the count reaches 0, the object is freed **immediately** (deterministic, like `shared_ptr`).
2. **Cyclic garbage collector (backup).** Reference counting can't free **cycles** (a → b → a). The `gc` module periodically finds unreachable cycles. It is **generational** (gen 0/1/2): new objects are checked often, survivors less often, based on the hypothesis that most objects die young.

Also: small objects come from the **pymalloc** arena allocator, and small ints (−5..256) and some strings are cached/interned.
*Trading angle:* GC pauses are **non-deterministic latency**, which is one reason latency-critical paths are written in C++. You can `gc.disable()` / `gc.freeze()` in hot loops.
</details>

In [ ]:
import sys, gc
a = []
print("refcount:", sys.getrefcount(a))    # +1 for the temporary argument reference
b = a
print("after b = a:", sys.getrefcount(a))

class Node: pass
x, y = Node(), Node()
x.other, y.other = y, x                    # a reference cycle
del x, y                                   # refcounts never hit 0...
print("collected by cyclic GC:", gc.collect(), "objects")
print("GC thresholds (gen0, gen1, gen2):", gc.get_threshold())

**Q5. ⭐ What is a generator? What does `yield` do?**

<details><summary>Answer</summary>

A generator is a function containing `yield`. Calling it returns a **generator object** (an iterator) without running the body. Each `next()` runs until the next `yield`, **hands out the value and pauses, keeping all local state**, then resumes from there on the next call. When the function returns, `StopIteration` is raised.
**Why:** lazy evaluation, **O(1) memory** for huge or infinite streams (read a 50 GB tick file line by line), and composable pipelines.
Related: generator expressions `(x*x for x in xs)`; `yield from sub_gen` delegates to another generator; `send()` lets you push values in (the historical basis of coroutines).
</details>

**Q6. Iterator vs iterable?**

<details><summary>Answer</summary>

An **iterable** has `__iter__()` returning an iterator (lists, dicts, strings, files). An **iterator** has `__next__()` (and `__iter__` returning itself) and remembers its position. It's single-use: once exhausted, it stays exhausted. A `for` loop calls `iter()` once, then `next()` until `StopIteration`.
</details>

**Q7. ⭐ What is a decorator?**

<details><summary>Answer</summary>

A function that **takes a function and returns a new function** that usually wraps the original with extra behaviour. `@timed` above `def f` is syntax sugar for `f = timed(f)`. It works because functions are first-class objects, and closures let the wrapper remember `fn`.
Uses: logging, timing, caching (`@functools.cache`), retries, auth checks, registering handlers. Use `@functools.wraps(fn)` so the wrapper keeps the original name and docstring. A decorator *with arguments* (`@retry(3)`) is a function that returns a decorator.
</details>

**Q8. ⭐ What is a context manager? How do you write one?**

<details><summary>Answer</summary>

An object used with `with` that guarantees **setup and teardown**, even when an exception occurs. It implements `__enter__` (setup, and its return value is bound by `as`) and `__exit__(exc_type, exc, tb)` (cleanup; returning `True` swallows the exception).
The easy way: `@contextlib.contextmanager` on a generator, where code before `yield` is enter and code after (in a `finally`) is exit.
Examples: `with open(...)`, `with lock:`, DB transactions, timers. **It's Python's version of C++ RAII**, which is a nice link to make in the interview.
</details>

**Q9. ⭐ How do you open and read a file in Python?**

<details><summary>Answer</summary>

```python
with open("data.csv", "r", encoding="utf-8") as f:
    for line in f:              # lazy, line by line
        process(line.rstrip("\n"))
```
`with` closes the file even on errors. Modes: `"r"`, `"w"` (truncate), `"a"` (append), `"rb"`/`"wb"` (binary), `"x"` (create, fail if exists). `f.read()` loads everything (careful with big files), `f.readlines()` gives a list, and `json.load(f)` / `csv.reader(f)` for structured data.
</details>

**Q10. ⭐ List vs tuple?**

<details><summary>Answer</summary>

- **List**: mutable, dynamic array, not hashable. Use for homogeneous collections that change.
- **Tuple**: immutable, **hashable** if its elements are, so it can be a dict key or set member (e.g. `(row, col)`). Slightly smaller and faster to create, and the compiler can constant-fold tuple literals. Use for fixed records or multiple return values.
Gotcha: a tuple containing a list is immutable *as a container*, but the list inside can still change, so the tuple isn't hashable.
</details>

**Q11. ⭐ List vs array (`array.array` / NumPy array)?**

<details><summary>Answer</summary>

A Python `list` stores **pointers to boxed objects** that can have mixed types. Flexible but memory-heavy (each `int` is a ~28-byte object) and cache-unfriendly.
`array.array('d')` and **NumPy arrays** store **raw, contiguous, fixed-type values** (like `std::vector<double>`): compact, cache-friendly, and NumPy runs **vectorised C loops** without the interpreter overhead, often 10–100× faster for numeric work. That's why quant Python code is NumPy/pandas-heavy.
</details>

**Q12. ⭐ How does a Python dict work? What is hashing?**

<details><summary>Answer</summary>

A dict is a **hash table** with **open addressing**. `hash(key)` picks a slot; collisions probe other slots using a perturbation scheme. Lookup, insert and delete are **O(1) average**, O(n) worst case. It resizes when about 2/3 full. Since 3.7 it keeps **insertion order** (a compact array of entries plus a sparse index table).
Keys must be **hashable**: `__hash__` must be stable over the object's lifetime and consistent with `__eq__` (a == b ⇒ hash(a) == hash(b)). That's why mutable lists can't be keys.
*Go deeper:* string hashes are randomised per process (PYTHONHASHSEED) to prevent hash-flooding DoS attacks.
</details>

**Q13. Mutable vs immutable, and what does "pass by object reference" mean?**

<details><summary>Answer</summary>

Python passes **references to objects**. The function gets a new name bound to the same object. If the object is **mutable** (list, dict), in-place changes are visible to the caller. **Rebinding** the parameter (`x = ...`) never affects the caller. Immutable objects (int, str, tuple) can't be changed in place, so they *behave* as if passed by value.
</details>

**Q14. `is` vs `==`?**

<details><summary>Answer</summary>

`==` compares **values** (`__eq__`). `is` compares **identity** (same object, same `id()`). Use `is` only for singletons: `x is None`. `a = 256; b = 256; a is b` is `True` only because CPython caches small ints (an implementation detail you shouldn't rely on).
</details>

**Q15. Shallow copy vs deep copy?**

<details><summary>Answer</summary>

A **shallow copy** (`list(a)`, `a[:]`, `a.copy()`, `copy.copy`) makes a new outer container, but the elements are **the same objects**. A **deep copy** (`copy.deepcopy`) recursively copies everything. Classic bug: `grid2 = grid[:]` for a 2-D list shares the rows.
</details>

**Q16. What are `*args` and `**kwargs`?**

<details><summary>Answer</summary>

`*args` collects extra positional arguments into a **tuple**; `**kwargs` collects extra keyword arguments into a **dict**. At a call site, `f(*lst, **d)` unpacks them. They're used in decorators to forward any signature.
</details>

**Q17. What's the mutable default argument trap?**

<details><summary>Answer</summary>

Default values are evaluated **once, when the function is defined**, so `def f(x, acc=[])` shares one list across calls. Use `acc=None` and create the list inside the function.
</details>

**Q18. What are `__slots__`?**

<details><summary>Answer</summary>

Declaring `__slots__ = ("a", "b")` in a class replaces the per-instance `__dict__` with fixed storage slots. It **saves memory** (useful for millions of small objects like orders or ticks) and makes attribute access slightly faster, but you can't add new attributes dynamically.
</details>

**Q19. `@staticmethod` vs `@classmethod` vs instance method?**

<details><summary>Answer</summary>

An instance method gets `self`. A `@classmethod` gets `cls` (used for alternative constructors like `Order.from_json`, and it works with subclasses). A `@staticmethod` gets neither: it's a plain function namespaced in the class.
</details>

**Q20. What is duck typing?**

<details><summary>Answer</summary>

"If it walks like a duck…": Python cares whether an object **has the methods you call**, not its declared type. Any object with `__iter__` works in a `for` loop. The static equivalent is `typing.Protocol`. The C++ contrast is templates/concepts (compile time) vs virtual interfaces (runtime).
</details>

**Q21. ⭐ What is a key problem with using Python in finance?**

<details><summary>Answer</summary>

Name several, then give the balanced view:
1. **Speed and latency.** It's interpreted and dynamically typed, so pure-Python loops are ~10–100× slower than C++. **GC and refcounting cause non-deterministic pauses.**
2. **The GIL.** There's no true multi-threaded CPU parallelism.
3. **Floating-point precision.** `0.1 + 0.2 != 0.3`. For money and accounting use `decimal.Decimal`, or integer ticks/cents (common in exchange protocols like ITCH, which sends prices as fixed-point integers).
4. **Dynamic typing.** Some bugs only show up at runtime, which is risky for production trading systems (mitigate with type hints + mypy and tests).
5. Memory overhead of boxed objects.

**Balanced view:** Python dominates **research, backtesting and data analysis** (NumPy, pandas, fast iteration), while **low-latency execution** is in C++ (or Rust/Java). Many firms prototype in Python and productionise the hot path in C++. You can mention that your LOB project follows this pattern (Python + Rust).
</details>

In [ ]:
from decimal import Decimal, getcontext
print(0.1 + 0.2, 0.1 + 0.2 == 0.3)
print(Decimal("0.1") + Decimal("0.2"), Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))
print(Decimal(0.1))           # ⚠️ constructing from a float carries the float's error. Use strings!
# ITCH-style fixed point: price as an integer number of 1/10000 dollars
px_int = 1872500              # $187.2500
print(px_int / 10_000)

**Q22. What is monkey patching / how are attributes looked up (MRO)?**

<details><summary>Answer</summary>

Attribute lookup: instance `__dict__` → class → base classes in **MRO** order (C3 linearisation, see `Cls.__mro__`). Because classes are mutable objects, you can replace methods at runtime ("monkey patching"). It's handy in tests (`unittest.mock.patch`) and dangerous in production.
</details>

**Q23. List comprehension vs generator expression vs `map`?**

<details><summary>Answer</summary>

`[f(x) for x in xs]` builds the whole list (eager). `(f(x) for x in xs)` is lazy (O(1) memory). `map(f, xs)` is also lazy in Python 3. Comprehensions are usually the most readable choice and slightly faster than an equivalent `for` + `append` loop.
</details>

---
# 2 · Concurrency & asynchronous programming

**Q24. ⭐ What is asynchronous programming? How does `asyncio` work?**

<details><summary>Answer</summary>

Async = **cooperative concurrency on a single thread**. An **event loop** runs many **coroutines** (`async def`). When a coroutine hits `await` on something not ready yet (a network read, a sleep), it **yields control** back to the loop, which runs another coroutine that is ready. There are no OS threads and no preemption. Tasks switch only at `await` points, so there are fewer race conditions.
**Best for:** many concurrent I/O-bound operations (thousands of sockets, websocket market-data feeds, REST calls). **Bad for:** CPU-bound work, which blocks the whole loop. Offload that with `loop.run_in_executor` or `asyncio.to_thread`.
Key API: `asyncio.run(main())`, `await`, `asyncio.gather(*coros)`, `asyncio.create_task`, `asyncio.Queue`.
</details>

In [ ]:
import asyncio, time

async def fetch(sym, delay):
    await asyncio.sleep(delay)          # stands in for a network call; yields to the event loop
    return f"{sym} quote after {delay}s"

async def main():
    t0 = time.perf_counter()
    results = await asyncio.gather(fetch("AAPL", 0.3), fetch("MSFT", 0.3), fetch("GOOG", 0.3))
    print(results, f"total {time.perf_counter() - t0:.2f}s (≈0.3, concurrent)")

# In Jupyter an event loop is already running, so use `await` directly.
# In a script you would write: asyncio.run(main())
await main()

**Q25. Concurrency vs parallelism?**

<details><summary>Answer</summary>

**Concurrency** = dealing with many tasks at once by *interleaving* them (possible on one core: asyncio, threads with the GIL). **Parallelism** = *executing* many tasks at the same instant on multiple cores (multiprocessing, C++ threads). Concurrency is about structure; parallelism is about execution.
</details>

**Q26. Threads vs asyncio for I/O: which one and why?**

<details><summary>Answer</summary>

Both work for I/O. **asyncio** scales to many thousands of connections cheaply (a coroutine is a few KB; a thread has an OS stack of around 8 MB virtual), has explicit switch points (easier to reason about races), but needs async-aware libraries all the way down. **Threads** work with any blocking library but cost more and need locks.
</details>

**Q27. ⭐ What is a race condition? How do you prevent it?**

<details><summary>Answer</summary>

When the result depends on the **timing/interleaving** of threads accessing shared data, and at least one of them writes. Example: two threads doing `count += 1` (read, add, write) lose updates. Even in Python with the GIL, `+=` on a shared variable is not atomic, because a thread switch can happen between bytecodes.
Prevent it with **mutual exclusion** (locks/mutexes), **atomic operations**, **immutable data**, **message passing** (queues), or **thread confinement** (each thread owns its data).
</details>

**Q28. ⭐ Deadlock: what is it, and what are the four conditions?**

<details><summary>Answer</summary>

Two or more threads each wait for a resource the other holds, forever. **Coffman conditions** (all four are needed): 1) **mutual exclusion**, 2) **hold and wait**, 3) **no preemption**, 4) **circular wait**.
Prevention: **global lock ordering** (breaks circular wait, the most practical fix), `try_lock` with timeout/back-off, acquiring all locks at once (`std::scoped_lock(m1, m2)` in C++17), or avoiding shared locks altogether.
</details>

**Q29. ⭐ Mutex vs semaphore vs spinlock vs condition variable?**

<details><summary>Answer</summary>

- **Mutex:** one owner at a time. The thread that locks it must unlock it.
- **Semaphore:** a counter allowing **N** concurrent holders (e.g. a pool of 5 DB connections). Binary semaphore ≈ mutex, but without ownership.
- **Spinlock:** busy-waits in a loop instead of sleeping. Good only when the critical section is **tiny** and the threads are on different cores (no context-switch cost). Common in **low-latency trading** with pinned cores.
- **Condition variable:** lets a thread **sleep until a condition becomes true** (the producer–consumer pattern); always used with a mutex and a predicate loop (spurious wakeups).
</details>

**Q30. What is the producer–consumer pattern?**

<details><summary>Answer</summary>

Producers push work items into a **bounded queue** and consumers pop them. It decouples speed differences and provides back-pressure. In Python: `queue.Queue` (thread-safe), `asyncio.Queue`, `multiprocessing.Queue`. In low-latency C++: **lock-free single-producer/single-consumer ring buffers** (e.g. market-data thread → strategy thread).
</details>

---
# 3 · Operating systems: processes, threads, memory, real-time
Reported: virtual memory, process management, page tables, real-time systems, dangling pointers.

**Q31. ⭐ Process vs thread?**

<details><summary>Answer</summary>

A **process** is a running program with its **own virtual address space**, file descriptors and resources. It's isolated: a crash doesn't affect other processes. A **thread** is a unit of execution **inside** a process. Threads **share** the heap, globals and file descriptors, but each has its **own stack and registers**.
Tradeoffs: threads are cheaper to create and switch between, and communicate through shared memory (fast, but they need synchronisation). Processes are safer and more isolated, but need **IPC** (pipes, sockets, shared memory) to communicate.
</details>

**Q32. ⭐ What is a context switch? Why is it expensive?**

<details><summary>Answer</summary>

The OS saves the running thread's state (registers, program counter, stack pointer) and restores another's. It's expensive because of the **kernel entry**, the scheduler's work, and above all the **indirect costs**: CPU caches and the **TLB get polluted or flushed** (on a process switch the address space changes). Direct cost is roughly 1–5 µs, plus cache warm-up.
*Trading angle:* low-latency systems **pin threads to isolated cores** (`isolcpus`, CPU affinity) and busy-poll to avoid context switches entirely.
</details>

**Q33. ⭐ What is virtual memory, and why do we use it?**

<details><summary>Answer</summary>

Each process sees its own large, contiguous **virtual address space**. The **MMU** (hardware) translates virtual addresses to **physical** ones using **page tables** managed by the OS. Memory is split into fixed-size **pages** (usually 4 KB) that map to physical **frames**.
Benefits:
1. **Isolation/protection.** A process can't touch another's memory, and pages carry permissions (read/write/execute).
2. **Simplicity.** Every program can be linked as if it owns the whole address space.
3. **Overcommit / paging to disk.** You can use more memory than physically exists (swap).
4. **Sharing and efficiency.** Shared libraries map once; `fork()` uses **copy-on-write**; `mmap` maps files into memory; allocation is lazy (a page is allocated on first touch).
</details>

**Q34. ⭐ What is a page table? How does address translation work?**

<details><summary>Answer</summary>

A per-process data structure mapping **virtual page number → physical frame number** plus flag bits (present, read/write, user/kernel, dirty, accessed).
A virtual address = **[page number | offset]**. The MMU looks up the page number to get the frame, then physical address = frame base + offset.
A flat table would be huge (a 48-bit space with 4 KB pages has 2³⁶ entries), so x86-64 uses a **multi-level (4 or 5 level) page table**: a tree where only the used parts are allocated. The CR3 register points at the root for the current process.
</details>

**Q35. ⭐ What is the TLB?**

<details><summary>Answer</summary>

The **Translation Lookaside Buffer** is a small, fast hardware cache of recent virtual→physical translations. A TLB **hit** gives a near-free translation. A **miss** requires a **page walk** through the multi-level table (several memory accesses). It's often flushed on a process context switch (unless tagged with PCID/ASID).
**Huge pages** (2 MB / 1 GB) cover more memory per TLB entry, so there are fewer misses. That's a common tuning step for **low-latency trading** systems and big in-memory order books.
</details>

**Q36. ⭐ What is a page fault? Minor vs major?**

<details><summary>Answer</summary>

The CPU accesses a virtual page that isn't currently mapped or is disallowed → trap into the kernel.
- **Minor (soft):** the page is in RAM but not mapped yet (first touch of lazily allocated memory, copy-on-write, a shared library page). Cheap-ish (≈ µs).
- **Major (hard):** the page must be **read from disk** (swap or a memory-mapped file). Very slow (≈ ms).
- **Invalid:** the address isn't legal, so the kernel sends **SIGSEGV** (segmentation fault).
*Trading angle:* pre-fault and lock memory at startup (`mlock`/`mlockall`, touch every page) so there are no faults on the hot path.
</details>

**Q37. What is thrashing? What are page replacement algorithms?**

<details><summary>Answer</summary>

**Thrashing:** the working set doesn't fit in RAM, so the system spends most of its time swapping pages in and out instead of doing work.
Replacement policies: **LRU** (ideal but costly to track exactly), **Clock/second-chance** (LRU approximation with an accessed bit, which is what real OSes use), FIFO (suffers from Belady's anomaly), and Optimal (theoretical: evict the page used farthest in the future).
</details>

**Q38. ⭐ Stack vs heap memory?**

<details><summary>Answer</summary>

- **Stack:** per-thread, LIFO, automatic lifetime (function locals). Allocation is just moving the stack pointer, so it's extremely fast and cache-hot. Size is limited (≈ 8 MB default on Linux), and overflowing it (deep recursion) crashes the program.
- **Heap:** dynamic allocation (`new`/`malloc`), with lifetime you control. Slower (allocator bookkeeping, possible locks, fragmentation) and prone to leaks or dangling pointers if managed manually.
*Trading angle:* avoid heap allocation on the hot path. Pre-allocate, use object pools, arena allocators, and `reserve()`.
</details>

**Q39. What does a process's memory layout look like?**

<details><summary>Answer</summary>

From low to high addresses: **text** (code, read-only) → **data** (initialised globals) → **BSS** (zero-initialised globals) → **heap** (grows up) → … memory-mapped regions and shared libraries … → **stack** (grows down) → kernel space (top, inaccessible from user mode).
</details>

**Q40. What happens in `fork()` and `exec()`?**

<details><summary>Answer</summary>

`fork()` creates a child process that is a **copy** of the parent. It returns 0 in the child and the child's PID in the parent. Memory is shared **copy-on-write**: pages are copied only when one side writes to them. `exec()` **replaces** the current process image with a new program. A shell runs a command with fork + exec + `wait()`.
</details>

**Q41. Zombie vs orphan process?**

<details><summary>Answer</summary>

**Zombie:** the child exited, but the parent hasn't called `wait()` to collect its exit status, so its process-table entry lingers (shown as `<defunct>`). **Orphan:** the parent died first. The child is re-parented to `init`/`systemd` (PID 1), which reaps it.
</details>

**Q42. User mode vs kernel mode; what is a system call?**

<details><summary>Answer</summary>

The CPU runs user code in **unprivileged mode**. Privileged operations (I/O, memory mapping, creating processes) require a **system call** (`read`, `write`, `mmap`, `fork`), which traps into **kernel mode**, runs kernel code and returns. Syscalls cost roughly 100 ns to 1 µs plus cache effects, which is why HFT uses **kernel bypass** (see Networking).
</details>

**Q43. ⭐ What are the process states? What does the scheduler do?**

<details><summary>Answer</summary>

States: **new → ready → running → (blocked/waiting on I/O) → ready → … → terminated**. The scheduler picks which ready thread runs next on each core. Linux's default is **CFS/EEVDF** (fair share based on virtual runtime). Alternatives: round-robin with a time slice, and priority scheduling.
Real-time policies on Linux: `SCHED_FIFO` and `SCHED_RR` (fixed priorities that always preempt normal tasks).
</details>

**Q44. ⭐ What is a real-time system? Hard vs soft real-time?**

<details><summary>Answer</summary>

A system where correctness depends on **meeting timing deadlines**, not just producing the right answer. The focus is **predictability/determinism** (worst-case latency), not average speed.
- **Hard real-time:** missing a deadline is a **failure** (airbag controller, pacemaker, flight control). Needs worst-case execution-time analysis and an **RTOS** (FreeRTOS, VxWorks, QNX).
- **Soft real-time:** occasional misses degrade quality but aren't catastrophic (video streaming, audio, most **trading systems**, where a slow order loses money but nothing explodes).
- (**Firm:** a late result is useless but not catastrophic.)
Techniques: priority-based **preemptive** scheduling (Rate-Monotonic, Earliest-Deadline-First), bounded interrupt latency, no unbounded operations (no GC, no dynamic allocation, no page faults on critical paths), and priority-inheritance locks.
</details>

**Q45. ⭐ What is priority inversion?**

<details><summary>Answer</summary>

A **low-priority** task holds a lock that a **high-priority** task needs, while a **medium-priority** task preempts the low one, so the high-priority task effectively waits on the medium one. Famous case: the **Mars Pathfinder** resets (1997).
Fixes: **priority inheritance** (the lock holder temporarily inherits the waiter's priority) or a **priority ceiling**.
</details>

**Q46. ⭐ What are CPU caches and cache locality? What is false sharing?**

<details><summary>Answer</summary>

Caches (L1 ~1 ns, L2 ~4 ns, L3 ~10–40 ns, RAM ~100 ns) store **64-byte cache lines**. **Spatial locality** (contiguous access, like arrays) and **temporal locality** (reuse) make code much faster. That's why a `std::vector` usually beats a `std::list` even for inserts in the middle at moderate sizes.
**False sharing:** two threads write **different variables that sit on the same cache line**, so the line ping-pongs between cores and performance collapses. The fix is padding/alignment: `alignas(64)` or `std::hardware_destructive_interference_size`.
</details>

**Q47. What is IPC? Name some mechanisms.**

<details><summary>Answer</summary>

Inter-process communication: **pipes** (`|`, anonymous or named FIFOs), **sockets** (Unix domain or TCP/UDP), **shared memory** (`shm_open` + `mmap`, the fastest, needs synchronisation), **message queues**, **signals**, **memory-mapped files**. Low-latency systems often use a shared-memory ring buffer between processes.
</details>

**Q48. What is a signal? What do Ctrl-C and Ctrl-Z do?**

<details><summary>Answer</summary>

An asynchronous notification sent to a process. **Ctrl-C** sends `SIGINT` (interrupt, usually terminates). **Ctrl-Z** sends `SIGTSTP` (suspend/stop the job). `kill PID` sends `SIGTERM` (polite, can be caught); `kill -9` sends `SIGKILL` (can't be caught or ignored). `SIGSEGV` means an invalid memory access. `SIGCHLD` goes to a parent when a child exits.
</details>

---
# 4 · Networking
Reported: TCP vs UDP.

**Q49. ⭐ TCP vs UDP?**

<details><summary>Answer</summary>

| | TCP | UDP |
|---|---|---|
| Connection | connection-oriented (3-way handshake) | connectionless |
| Reliability | guaranteed delivery, retransmits, **in-order** | best effort: packets may be lost, duplicated or reordered |
| Flow/congestion control | yes | no |
| Data model | byte **stream** (no message boundaries) | **datagrams** (message boundaries kept) |
| Header | 20+ bytes | 8 bytes |
| Latency | higher (ACKs, retransmit, head-of-line blocking) | lower, predictable |
| Use | HTTP, SSH, DB connections, **order entry** (FIX/OUCH) | DNS, video/voice, games, **market-data multicast** |
</details>

**Q50. ⭐ Why do exchanges publish market data over UDP multicast but take orders over TCP?**

<details><summary>Answer</summary>

**Market data (UDP multicast):** one packet reaches **thousands of subscribers at once**, which is efficient and fair. It has minimal latency and no per-client connection state, and **a retransmit is useless anyway**, since a stale quote has no value by the time it's resent. Gaps are detected with **sequence numbers**, and recovered via a separate TCP retransmission/snapshot service. NASDAQ **ITCH** over **MoldUDP64** is exactly this, and it's the data your LOB project parses.
**Orders (TCP):** every order must arrive **exactly once, in order, reliably** (e.g. OUCH and FIX run over TCP).
This question ties straight into your project, so bring it up if they ask about UDP.
</details>

**Q51. ⭐ Explain the TCP 3-way handshake (and teardown).**

<details><summary>Answer</summary>

**SYN** (client, with its initial seq number x) → **SYN-ACK** (server: seq y, ack x+1) → **ACK** (client: ack y+1). Both sides now agree on initial sequence numbers.
Teardown: **FIN → ACK, FIN → ACK** (4-way, because each direction closes independently). The side that closes first waits in **TIME_WAIT** (2×MSL) so stray packets don't corrupt a new connection.
</details>

**Q52. How does TCP guarantee reliability? What are flow and congestion control?**

<details><summary>Answer</summary>

Reliability: **sequence numbers**, **ACKs**, **retransmission** on timeout or on 3 duplicate ACKs (fast retransmit), checksums, and reordering at the receiver.
**Flow control:** the receiver advertises a **window** (how much buffer it has left) so a fast sender doesn't overwhelm a slow receiver.
**Congestion control:** the sender limits itself to avoid overloading the **network**: slow start, AIMD (additive increase, multiplicative decrease), and algorithms like CUBIC and BBR.
</details>

**Q53. ⭐ What is Nagle's algorithm, and why do trading systems disable it?**

<details><summary>Answer</summary>

Nagle **buffers small writes** until the previous packet is ACKed, to avoid sending many tiny packets. That adds latency (worse combined with delayed ACKs). Latency-sensitive apps set **`TCP_NODELAY`** to send immediately.
</details>

**Q54. What is kernel bypass?**

<details><summary>Answer</summary>

The normal network path goes NIC → kernel TCP/IP stack → syscall → user space, which costs several µs and adds jitter. **Kernel bypass** (Solarflare **OpenOnload/ef_vi**, **DPDK**, RDMA) lets the application read and write the NIC's buffers **directly from user space**, often with **busy-polling**, cutting latency to around 1 µs. HFT goes further with **FPGAs** that handle feed parsing and even order generation in hardware.
</details>

**Q55. OSI / TCP-IP layers?**

<details><summary>Answer</summary>

TCP/IP model: **Link** (Ethernet, MAC addresses) → **Internet** (IP, routing, IP addresses) → **Transport** (TCP/UDP, ports) → **Application** (HTTP, DNS, FIX).
OSI's 7 layers: Physical, Data Link, Network, Transport, Session, Presentation, Application ("Please Do Not Throw Sausage Pizza Away").
</details>

**Q56. What happens when you type a URL into a browser?**

<details><summary>Answer</summary>

**DNS** resolution (browser cache → OS → resolver → root → TLD → authoritative server) → **TCP handshake** (or QUIC/UDP for HTTP/3) → **TLS handshake** (certificate check, key exchange) → **HTTP request** → the server (load balancer → app → DB) builds a response → the browser parses HTML, fetches CSS/JS/images, and renders.
</details>

**Q57. What is a socket? How does a TCP server work?**

<details><summary>Answer</summary>

A socket is an OS endpoint for network communication (IP + port + protocol), accessed through a file descriptor. Server: `socket()` → `bind()` → `listen()` → `accept()` (returns a new socket per client) → `recv()`/`send()` → `close()`. Client: `socket()` → `connect()` → send/recv. To handle many clients: threads, or **I/O multiplexing** (`select`/`poll`/**`epoll`**), which is what asyncio uses underneath.
</details>

**Q58. Multicast vs broadcast vs unicast?**

<details><summary>Answer</summary>

**Unicast:** one to one. **Broadcast:** one to everyone on the subnet. **Multicast:** one to a **subscribed group** (receivers join the group via IGMP, and routers replicate packets only where needed). That's why exchange feeds use multicast.
</details>

**Q59. HTTP vs WebSocket?**

<details><summary>Answer</summary>

HTTP is request–response, stateless, and the client always initiates. A **WebSocket** starts as an HTTP upgrade and then becomes a **persistent, full-duplex** connection: the server can push updates any time. That's the usual choice for streaming prices to a browser or trading UI.
</details>

---
# 5 · Linux / Unix
Reported: print the working directory, move a job to the foreground, find a word in a file.

| Task | Command |
|---|---|
| ⭐ print working directory | `pwd` |
| list files (long, hidden, human sizes) | `ls -lah` |
| ⭐ find a word in a file | `grep "word" file` · `-i` ignore case · `-n` line numbers · `-r` recursive · `-w` whole word · `-v` invert · `-c` count · `-E` extended regex |
| find files by name | `find . -name "*.py"` · `find . -type f -mtime -1` (modified in the last day) |
| ⭐ jobs: suspend / list / foreground / background | `Ctrl-Z` · `jobs` · **`fg %1`** (bring to foreground) · `bg %1` (resume in background) |
| run in background / survive logout | `cmd &` · `nohup cmd &` · `disown` · `tmux`/`screen` |
| processes | `ps aux` · `ps -ef \| grep python` · `top` / `htop` · `pgrep -f name` |
| kill | `kill PID` (SIGTERM) · `kill -9 PID` (SIGKILL) · `pkill -f name` |
| view files | `cat` · `less` · `head -n 20` · `tail -n 50` · ⭐ **`tail -f app.log`** (follow a live log) |
| count lines/words | `wc -l file` |
| top frequency | `sort file \| uniq -c \| sort -rn \| head` |
| columns | `cut -d',' -f2 file.csv` · `awk -F',' '{sum += $3} END {print sum}' f.csv` |
| replace text | `sed -i 's/old/new/g' file` |
| permissions | `chmod 755 script.sh` · `chmod +x` · `chown user:group f` (r=4, w=2, x=1; user/group/other) |
| disk | `df -h` (filesystem) · `du -sh dir` (directory size) |
| network | `ss -tulpn` / `netstat -tulpn` (listening ports) · `lsof -i :8080` · `curl` · `ping` · `traceroute` |
| open files of a process | `lsof -p PID` |
| env / path | `echo $PATH` · `export VAR=1` · `which python` |
| remote | `ssh user@host` · `scp f user@host:/path` · `rsync -av` |
| pipes & redirects | `a \| b` · `> out` (overwrite) · `>> out` (append) · `2> err` · `2>&1` · `< in` |
| system / perf | `free -h` · `uptime` · `vmstat 1` · `strace -p PID` · `perf top` |

**Q60. ⭐ How do you find all lines containing ERROR in every log file in a directory tree, with line numbers, case-insensitive?**

<details><summary>Answer</summary>

`grep -rni "error" /var/log/myapp/`, or restricted to `.log` files: `grep -rni --include="*.log" "error" .`
Count per file: `grep -rci "error" .` · Only file names: `grep -rli "error" .`
</details>

**Q61. ⭐ You started a long job in the foreground. How do you move it to the background, and back?**

<details><summary>Answer</summary>

`Ctrl-Z` (suspend) → `bg` (resume in background) → `jobs` (list) → `fg %1` (bring job 1 back to the **foreground**). To start it in the background: `cmd &`. To keep it running after logout: `nohup cmd > out.log 2>&1 &` or run it inside `tmux`.
</details>

**Q62. What's the difference between a hard link and a soft (symbolic) link?**

<details><summary>Answer</summary>

A **hard link** is another directory entry pointing to the **same inode** (same data). It can't cross filesystems or point to directories, and the data lives until the last link is removed. A **symlink** is a small file containing a **path**. It can cross filesystems, and it breaks if the target is deleted.
</details>

**Q63. What's an inode? What's a file descriptor?**

<details><summary>Answer</summary>

An **inode** stores a file's metadata (size, owner, permissions, timestamps, pointers to data blocks), everything except its name. A **file descriptor** is a small integer a process uses to refer to an open file, socket or pipe (0 = stdin, 1 = stdout, 2 = stderr). "Everything is a file" in Unix.
</details>

**Q64. How would you find which process is using port 8080 and kill it?**

<details><summary>Answer</summary>

`lsof -i :8080` or `ss -ltnp | grep 8080` → get the PID → `kill PID` (then `kill -9 PID` if it ignores SIGTERM).
</details>

**Q65. How do you see the 10 IP addresses with the most requests in an access log?**

<details><summary>Answer</summary>

`awk '{print $1}' access.log | sort | uniq -c | sort -rn | head -10`. This `sort | uniq -c | sort -rn` pattern comes up a lot.
</details>

---
# 6 · C++
Your résumé has C++, and "C++ internals, smart pointers & data structures" was reported. At a trading firm this section matters a lot.

**Q66. ⭐ Pointer vs reference?**

<details><summary>Answer</summary>

A **pointer** is an object holding an address. It can be null, can be reseated to point elsewhere, supports arithmetic, and needs `*`/`->` to dereference. A **reference** is an **alias** for an existing object. It must be initialised, can't be null (in valid code), can't be reseated, and uses plain syntax.
Guideline: use references for "must exist" parameters (`const T&` to avoid copies), and pointers (or `std::optional`/smart pointers) when "may be absent" or ownership is involved.
</details>

**Q67. ⭐ What is RAII?**

<details><summary>Answer</summary>

**Resource Acquisition Is Initialization**: tie a resource's lifetime (memory, file, lock, socket) to an **object's lifetime**. Acquire in the constructor, **release in the destructor**. Because destructors run automatically when objects go out of scope, **including during exception unwinding**, you get no leaks and exception safety.
Examples: `std::vector`, `std::unique_ptr`, `std::lock_guard`, `std::fstream`. (Python's equivalent is the `with` statement.)
</details>

**Q68. ⭐ Explain `unique_ptr`, `shared_ptr` and `weak_ptr`.**

<details><summary>Answer</summary>

- **`std::unique_ptr<T>`**: **exclusive ownership**. It can't be copied, **only moved** (`std::move`). Zero overhead compared with a raw pointer (with the default deleter). Deletes the object when it goes out of scope. **This should be the default.**
- **`std::shared_ptr<T>`**: **shared ownership** via **reference counting**. Copying increments an **atomic** count, and the object is deleted when the last owner goes away. Costs: a **control block** (strong count, weak count, deleter), atomic inc/dec on every copy (slow under contention), and two pointers' worth of size.
- **`std::weak_ptr<T>`**: a **non-owning** observer of a `shared_ptr`-managed object. It doesn't keep the object alive. Call `lock()` to get a `shared_ptr` (null if the object is already gone). Used to **break reference cycles** (e.g. child → parent back-pointers, observers, caches).
Create them with **`std::make_unique` / `std::make_shared`**: exception-safe, and `make_shared` does **one allocation** for the object and control block together.
</details>

In [ ]:
run_cpp(r"""
#include <iostream>
#include <memory>
struct Order {
    int id;
    explicit Order(int i) : id(i) { std::cout << "  Order " << id << " created\n"; }
    ~Order() { std::cout << "  Order " << id << " destroyed\n"; }
};
int main() {
    std::cout << "unique_ptr:\n";
    {
        auto u = std::make_unique<Order>(1);
        // auto u2 = u;              // ❌ compile error: can't copy
        auto u2 = std::move(u);      // ✅ transfer ownership; u is now null
        std::cout << "  u is " << (u ? "set" : "null") << "\n";
    }                                // Order 1 destroyed here

    std::cout << "shared_ptr:\n";
    std::weak_ptr<Order> w;
    {
        auto s1 = std::make_shared<Order>(2);
        auto s2 = s1;
        w = s1;
        std::cout << "  use_count = " << s1.use_count() << "\n";   // 2 (weak doesn't count)
    }                                // last owner gone -> destroyed
    std::cout << "  weak expired? " << std::boolalpha << w.expired() << "\n";
}
""")

**Q69. ⭐ How can `shared_ptr` leak memory? How do you fix it?**

<details><summary>Answer</summary>

**Reference cycles:** A holds a `shared_ptr` to B, and B holds a `shared_ptr` to A. When outside owners disappear, each count stays at 1 forever, so neither is destroyed (a leak). Fix: make one direction a **`weak_ptr`** (usually the back-pointer: child → parent). Also watch for capturing `shared_from_this()` in long-lived lambdas and callbacks.
</details>

In [ ]:
run_cpp(r"""
#include <iostream>
#include <memory>
struct B;
struct A { std::shared_ptr<B> b; ~A(){ std::cout << "~A\n"; } };
struct B { std::shared_ptr<A> a;   ~B(){ std::cout << "~B\n"; } };        // cycle!
struct B2;
struct A2 { std::shared_ptr<B2> b; ~A2(){ std::cout << "~A2\n"; } };
struct B2 { std::weak_ptr<A2> a;   ~B2(){ std::cout << "~B2\n"; } };      // weak back-pointer
int main() {
    { auto a = std::make_shared<A>(); auto b = std::make_shared<B>(); a->b = b; b->a = a; }
    std::cout << "(no destructors above = leaked)\n";
    { auto a = std::make_shared<A2>(); auto b = std::make_shared<B2>(); a->b = b; b->a = a; }
    std::cout << "(both destroyed with weak_ptr)\n";
}
""")

**Q70. ⭐ What is a dangling pointer? Give examples. How do you avoid it?**

<details><summary>Answer</summary>

A pointer or reference to memory that has been **freed or gone out of scope**. Using it is **undefined behaviour**: it might crash, might silently read garbage, or might "work" until production.
Examples:
1. Returning a pointer/reference to a **local variable**.
2. Using a pointer after `delete`/`free` (**use-after-free**). A **double delete** is related.
3. **Iterator/pointer invalidation**: keeping `&v[0]` or an iterator, then `v.push_back()` reallocates.
4. A `string_view`/`span` outliving the string/buffer it views.
5. A lambda capturing a local **by reference** that runs after the scope ends.
Avoid: RAII and smart pointers, don't return references to locals, set raw pointers to `nullptr` after delete (a partial fix), watch invalidation rules, use tools: **AddressSanitizer** (`-fsanitize=address`), Valgrind, compiler warnings.
</details>

In [ ]:
run_cpp(r"""
#include <iostream>
#include <vector>
int* bad() { int x = 42; return &x; }            // ⚠️ dangling: x dies at return
int main() {
    std::vector<int> v = {1, 2, 3};
    int* p = &v[0];
    std::cout << "capacity before: " << v.capacity() << "\n";
    v.push_back(4);                               // may reallocate -> p now dangles
    std::cout << "capacity after:  " << v.capacity()
              << "  (if it grew, p points to freed memory)\n";
    (void)p; (void)bad;
}
""")
# Try with AddressSanitizer on your own machine to see it caught:
#   g++ -std=c++20 -g -fsanitize=address demo.cpp && ./a.out

**Q71. Memory leak vs dangling pointer?**

<details><summary>Answer</summary>

A **leak** means memory is allocated and **never freed** (you lost the last pointer to it), so usage grows over time. A **dangling pointer** is the opposite problem: you still **use** memory **after** it was freed. Leaks waste memory; dangling pointers corrupt it (UB). RAII and smart pointers prevent both.
</details>

**Q72. ⭐ `new`/`delete` vs `malloc`/`free`?**

<details><summary>Answer</summary>

`new` allocates **and calls the constructor**, is type-safe, and throws `std::bad_alloc` on failure; `delete` calls the **destructor** and then frees. `malloc` only allocates raw bytes (returns `void*`, `NULL` on failure) and `free` only frees. Never mix them. Use `new[]` with `delete[]`. In modern C++, avoid raw `new`/`delete` and use `make_unique`, containers, etc.
</details>

**Q73. ⭐ What are move semantics? What does `std::move` actually do?**

<details><summary>Answer</summary>

Move semantics let an object **steal the resources** (heap buffer, file handle) of a temporary or no-longer-needed object instead of deep-copying them. For example, moving a `vector` copies 3 pointers instead of N elements.
It's implemented with **rvalue references** (`T&&`): the **move constructor** `T(T&& other) noexcept` and **move assignment**, which take the guts of `other` and leave it in a *valid but unspecified* state (usually empty).
**`std::move` doesn't move anything.** It's just a **cast to an rvalue reference** (`static_cast<T&&>`), which *allows* the move overload to be chosen. Don't use the moved-from object except to assign to it or destroy it.
</details>

**Q74. lvalue vs rvalue?**

<details><summary>Answer</summary>

Roughly: an **lvalue** has an identity/name and you can take its address (`x`, `arr[0]`, `*p`). An **rvalue** is a temporary without persistent identity (`42`, `x + y`, `std::string("hi")`, a function's return by value). `T&` binds to lvalues, `T&&` binds to rvalues, and `const T&` binds to both (which is why it's the classic read-only parameter type).
</details>

**Q75. ⭐ Rule of Three / Five / Zero?**

<details><summary>Answer</summary>

- **Rule of Three (C++98):** if a class needs a custom **destructor**, **copy constructor**, or **copy assignment**, it probably needs all three, because it manages a resource manually.
- **Rule of Five (C++11):** add the **move constructor** and **move assignment**.
- **Rule of Zero (preferred):** don't manage resources manually at all. Hold members like `std::vector`, `std::string` and `std::unique_ptr` that already manage themselves, and write **none** of the five; the compiler-generated ones are correct.
</details>

**Q76. Why should move constructors be `noexcept`?**

<details><summary>Answer</summary>

`std::vector` must keep the **strong exception guarantee** when it reallocates. It only *moves* elements into the new buffer if the move constructor is `noexcept`; otherwise it **copies** them (in case a move throws halfway). A missing `noexcept` can silently turn every reallocation into deep copies.
</details>

**Q77. What is copy elision / RVO?**

<details><summary>Answer</summary>

The compiler constructs a returned object **directly in the caller's storage**, skipping the copy/move. It's **guaranteed** in C++17 for returning prvalues (`return T(...)`) and commonly applied for named locals (NRVO). So returning a big `vector` by value is cheap. Don't write `return std::move(local);`, because that can *prevent* NRVO.
</details>

**Q78. ⭐ How do virtual functions work (vtable)? What does runtime polymorphism cost?**

<details><summary>Answer</summary>

Every class with virtual functions has a **vtable** (a static table of function pointers). Each object carries a hidden **vptr** pointing to its class's vtable. A virtual call `base->f()` loads the vptr, indexes the table and makes an **indirect call**, so the right derived override runs at **runtime**.
Costs: an extra pointer per object, one or two memory loads, an **indirect branch** that can be mispredicted, and above all **no inlining**. That matters in tight loops, so low-latency code often uses **static polymorphism (templates/CRTP)** or `final` to let the compiler devirtualise.
</details>

**Q79. ⭐ Why do base classes need a virtual destructor?**

<details><summary>Answer</summary>

If you `delete` a derived object **through a base pointer** and the base destructor isn't `virtual`, only the base destructor runs. That's **undefined behaviour**: in practice the derived members leak, derived cleanup is skipped. Rule: a class meant for polymorphic use (has any virtual function) needs a **`virtual ~Base() = default;`**.
</details>

**Q80. Static vs dynamic polymorphism? What is CRTP?**

<details><summary>Answer</summary>

**Dynamic:** virtual functions, dispatch resolved at **runtime** (flexible, heterogeneous containers, but has vtable costs). **Static:** templates/overloads, resolved at **compile time** (inlinable, zero overhead, but code bloat and all types known at compile time).
**CRTP** (Curiously Recurring Template Pattern): `class Derived : public Base<Derived>`. The base calls `static_cast<Derived*>(this)->impl()`, which gives interface-style reuse **without virtual calls**. It's popular in HFT code.
</details>

**Q81. ⭐ What does `const` mean in its different positions?**

<details><summary>Answer</summary>

`const int* p` / `int const* p`: pointer to const int (can't modify `*p`, can reseat `p`). `int* const p`: const pointer (can't reseat, can modify `*p`). `const int* const p`: both.
A **`const` member function** `int get() const` promises not to modify the object (it can be called on const objects). `mutable` members are exempt.
`constexpr`: evaluable at **compile time**. `consteval` (C++20): *must* be evaluated at compile time.
</details>

**Q82. What does `static` mean in C++?**

<details><summary>Answer</summary>

Four meanings: 1) a **static local variable** is initialised once and persists across calls (thread-safe init since C++11). 2) A **static data member** is shared by all instances. 3) A **static member function** has no `this`. 4) A **static global/function** has **internal linkage** (visible only in that translation unit).
</details>

**Q83. `struct` vs `class`?**

<details><summary>Answer</summary>

The only difference is **default access**: `struct` members and bases are public by default, `class` members are private. By convention, use `struct` for plain data aggregates and `class` for types with invariants.
</details>

**Q84. What is `inline`? Does it force inlining?**

<details><summary>Answer</summary>

No. Today `inline` mainly means "this definition may appear in multiple translation units" (it relaxes the one-definition rule, so functions can be defined in headers). Whether a call is actually inlined is the **optimizer's** decision (`-O2`). Compiler-specific attributes like `__attribute__((always_inline))` can force it.
</details>

**Q85. ⭐ What is undefined behaviour? Give examples.**

<details><summary>Answer</summary>

Code whose behaviour the standard doesn't define. The compiler may **assume it never happens** and optimise on that basis, so the effects can be bizarre and non-local.
Examples: out-of-bounds access, dereferencing null or dangling pointers, **signed integer overflow**, data races, use of an uninitialised variable, deleting through a non-virtual base destructor, violating strict aliasing, modifying a `const` object.
Tools: `-fsanitize=address,undefined`, `-Wall -Wextra`, Valgrind.
</details>

**Q86. ⭐ STL containers: when would you use each? Complexities?**

<details><summary>Answer</summary>

| Container | Structure | Lookup | Insert/erase | Notes |
|---|---|---|---|---|
| `vector` | contiguous dynamic array | O(1) index | O(1) amortized at end, O(n) middle | **default choice**; cache-friendly |
| `deque` | chunked array | O(1) index | O(1) both ends | no reallocation of all elements; pointers to elements stay valid when pushing at the ends |
| `list` | doubly linked list | O(n) | O(1) given an iterator | stable iterators, splicing; poor cache behaviour |
| `map` / `set` | **red-black tree** (sorted) | O(log n) | O(log n) | ordered iteration, `lower_bound` → **price levels of an order book** |
| `unordered_map` / `set` | hash table (separate chaining) | O(1) avg | O(1) avg | no order; rehash invalidates iterators → **order-ID → order lookup** |
| `priority_queue` | binary heap on `vector` | O(1) top | O(log n) | max-heap by default |
| `array` | fixed-size, on the stack | O(1) | — | size known at compile time |
</details>

**Q87. ⭐ How does `std::vector` grow? What are `reserve` and `emplace_back`?**

<details><summary>Answer</summary>

When `size == capacity`, `push_back` allocates a **bigger buffer (growth factor ~1.5× or 2×, depending on the library)**, moves/copies the elements across and frees the old buffer. That's why **amortised O(1)**, and why **all pointers, references and iterators are invalidated** on reallocation.
**`reserve(n)`** pre-allocates capacity, avoiding repeated reallocations (important for latency). `resize` changes the *size*.
**`emplace_back(args...)`** constructs the element **in place** from constructor arguments, while `push_back(obj)` copies/moves an existing object. Emplace can save a temporary.
</details>

**Q88. Iterator invalidation rules (the main ones)?**

<details><summary>Answer</summary>

- `vector`: reallocation invalidates **everything**. Insert/erase invalidates at and after the position.
- `deque`: insert in the middle invalidates everything; push at the ends invalidates iterators but **not references**.
- `list`/`map`/`set`: only iterators to the **erased** element are invalidated.
- `unordered_map`: **rehash** invalidates iterators (not references); erase invalidates only the erased element.
Classic bug: erasing inside a range-for loop. Use `it = v.erase(it)` or erase-remove / `std::erase_if` (C++20).
</details>

**Q89. ⭐ `map` vs `unordered_map`: which for an order book?**

<details><summary>Answer</summary>

Typically **both**:
- **Price levels** need **sorted order** (best bid = highest, best ask = lowest, walk levels in order) → `std::map<Price, Level>` (bids with `std::greater`) gives O(log n) insert and O(1) best via `begin()`. Many real implementations use a **flat array indexed by price tick** around the mid-price for O(1) access and cache friendliness.
- **Order ID → order** for O(1) cancel/modify → `unordered_map<OrderId, Order*>` (or a pre-sized flat array/hash map to avoid allocations).
- Each level holds a **FIFO queue** of orders (price-time priority), often an **intrusive doubly linked list** so a cancel is O(1) given the order pointer.
</details>

**Q90. How does `std::unordered_map` handle collisions? What is the load factor?**

<details><summary>Answer</summary>

The standard library uses **separate chaining** (each bucket is a linked list of nodes). **Load factor** = elements / buckets. When it exceeds `max_load_factor()` (default 1.0) the table **rehashes** (more buckets, every element redistributed): O(n), and it invalidates iterators. Use `reserve(n)` to avoid rehashing. Node-based storage is cache-unfriendly, so HFT code often uses **open-addressing flat hash maps** (Abseil `flat_hash_map`, `robin_hood`).
</details>

**Q91. What is a template? What are concepts?**

<details><summary>Answer</summary>

Templates are **compile-time generic code**: the compiler **instantiates** a separate version for each set of type arguments (zero runtime overhead, but more code). **Template specialisation** customises the code for particular types. **Concepts** (C++20) constrain template parameters (`template<std::integral T>`), giving clearer errors and overloading on requirements.
</details>

**Q92. ⭐ `volatile` vs `std::atomic`?**

<details><summary>Answer</summary>

`volatile` only tells the compiler **not to optimise away reads and writes** (for memory-mapped hardware registers, signal handlers). It gives **no atomicity and no ordering** between threads, so it is **not** for thread synchronisation in C++.
**`std::atomic<T>`** provides **indivisible** read-modify-write operations (`fetch_add`, `compare_exchange`) plus **memory-ordering** guarantees (`seq_cst` by default; `acquire`/`release`/`relaxed` for performance). It's the foundation of **lock-free** data structures (e.g. an SPSC ring buffer in trading systems).
</details>

**Q93. `std::thread`, `std::mutex`, `lock_guard` vs `unique_lock`, condition variables?**

<details><summary>Answer</summary>

`std::thread t(fn, args...)` must be `join()`ed or `detach()`ed before destruction (else `std::terminate`), or use `std::jthread` (C++20), which auto-joins. Protect shared data with a `std::mutex`, locked through **RAII**:
- `std::lock_guard<std::mutex>`: simplest scoped lock.
- `std::unique_lock`: can unlock/relock, defer, and move. It's **required with `std::condition_variable`**.
- `std::scoped_lock(m1, m2)`: locks several mutexes without deadlock.
Condition variable: `cv.wait(lock, [&]{ return !q.empty(); });` (the predicate handles spurious wakeups).
</details>

**Q94. What are memory orderings (acquire/release) in one sentence each?**

<details><summary>Answer</summary>

- **`relaxed`:** atomicity only, no ordering. Fine for counters/statistics.
- **`release`** (on a store): all my earlier writes become visible to whoever **acquires** this value.
- **`acquire`** (on a load): no later reads/writes of mine move before this load, so I see everything the releasing thread wrote before its release.
- **`seq_cst`** (default): one global order of all seq_cst operations. Simplest to reason about, and slightly slower on some architectures.
Classic pair: producer writes data, then `flag.store(true, release)`; consumer does `while (!flag.load(acquire));` and then reads the data safely.
</details>

**Q95. How is a C++ program built? (preprocessor → compiler → linker)**

<details><summary>Answer</summary>

1. **Preprocessor:** `#include` text pasting, macros, `#ifdef`.
2. **Compiler:** each `.cpp` (a **translation unit**) → assembly → **object file** (`.o`).
3. **Linker:** combines object files and libraries, resolves symbols → executable. **Static** libraries (`.a`) are copied in; **dynamic/shared** libraries (`.so`, `.dll`) are loaded at runtime.
Typical errors: "undefined reference" is a **linker** error (declared but never defined or linked); "multiple definition" is an ODR violation (a non-inline function defined in a header).
</details>

**Q96. Exceptions in C++: cost, `noexcept`, and why some trading code avoids them?**

<details><summary>Answer</summary>

Modern "zero-cost" exceptions cost **nothing on the happy path**, but **throwing is very slow** (stack unwinding, table lookups, allocation) and the timing is unpredictable. `noexcept` documents that a function won't throw and enables optimisations (e.g. vector moves). Many low-latency codebases **compile with `-fno-exceptions`** or never throw on the hot path, using error codes / `std::expected` (C++23) instead.
</details>

**Q97. ⭐ What makes C++ suitable for low-latency trading, and what do you avoid on the hot path?**

<details><summary>Answer</summary>

Why C++: **no GC**, deterministic destruction (RAII), control over **memory layout** and allocation, zero-cost abstractions (templates, inlining), direct access to hardware features (SIMD, atomics, cache alignment), and mature compilers and tooling.
Avoid on the hot path: **heap allocation** (pre-allocate, use pools/arenas), **locks** (use lock-free queues, single-writer designs), **virtual calls** in tight loops, **exceptions**, **system calls and I/O** (log asynchronously), **branch mispredictions** (`[[likely]]`, branchless code), **false sharing**, and **page faults** (pre-touch memory, huge pages). Measure with `perf` and use **cache-friendly data layouts** (struct-of-arrays, flat arrays).
</details>

**Q98. `std::string_view` / `std::span`: what are they and what's the danger?**

<details><summary>Answer</summary>

Non-owning, cheap **views** (a pointer plus a length) over contiguous characters or elements. They avoid copies when parsing (great for parsing messages from a network buffer). The danger is that they **don't extend lifetime**, so a view into a temporary string dangles: `std::string_view sv = std::string("hi");` is a bug.
</details>

**Q99. `std::optional`, `std::variant`, `std::any`?**

<details><summary>Answer</summary>

`optional<T>`: maybe-a-value, no heap allocation (better than a null pointer or sentinel values). `variant<A, B, C>`: a **type-safe union**; use `std::visit` for dispatch, which is a closed set of types without virtual calls (good for message types in a feed handler). `any`: holds any type, uses type erasure and possibly the heap, and is rarely what you want.
</details>

**Q100. What are lambdas? Capture by value vs reference?**

<details><summary>Answer</summary>

Anonymous function objects: `[capture](params) { body }`. `[=]` captures copies at creation; `[&]` captures references (they **dangle** if the lambda outlives the scope, e.g. callbacks or threads). `[x = std::move(buf)]` is an init capture. `mutable` allows modifying by-value captures. Each lambda has a unique type, so store it with `auto`, pass it to templates, or wrap it in `std::function` (type erasure, possible allocation, slower).
</details>

---
# 7 · Git

**Q101. ⭐ `git merge` vs `git rebase`?**

<details><summary>Answer</summary>

**Merge** combines branches with a **merge commit**. It keeps the true history and is non-destructive, but can clutter the history.
**Rebase** **replays your commits on top of** another branch (it creates **new commits** with new hashes), giving a **linear, clean history**.
**Golden rule:** never rebase commits that were **already pushed or shared**, because it rewrites history others depend on (force-push chaos). Typical flow: `git fetch && git rebase origin/main` on your private feature branch, then merge or squash-merge the PR.
`git rebase -i HEAD~3`: interactive (squash, reorder, edit, drop commits) to clean up before a PR.
</details>

**Q102. `git fetch` vs `git pull`?**

<details><summary>Answer</summary>

`fetch` downloads remote commits and updates remote-tracking branches (`origin/main`) **without touching** your working branch. `pull` = `fetch` + `merge` (or `rebase` with `--rebase`) into your current branch.
</details>

**Q103. `git reset` vs `git revert`?**

<details><summary>Answer</summary>

`revert` makes a **new commit that undoes** an earlier one. It's safe for shared branches. `reset` **moves the branch pointer** back (rewrites history): `--soft` keeps changes staged, `--mixed` (default) keeps them unstaged, `--hard` **discards** them. Use reset only on local, unpushed work.
</details>

**Q104. What are `git stash`, `cherry-pick` and `bisect`?**

<details><summary>Answer</summary>

`stash` temporarily shelves uncommitted changes (`git stash pop` restores them). `cherry-pick <sha>` applies one specific commit onto the current branch (e.g. a hotfix). `bisect` **binary-searches** history for the commit that introduced a bug (mark good/bad commits, O(log n) steps).
</details>

**Q105. How do you resolve a merge conflict?**

<details><summary>Answer</summary>

Git marks conflicting hunks with `<<<<<<<`, `=======` and `>>>>>>>`. Edit the file to the correct combined version, `git add` it, then `git commit` (merge) or `git rebase --continue`. Run the tests before continuing. `git merge --abort` / `git rebase --abort` bails out.
</details>

**Q106. What is HEAD? What is a detached HEAD?**

<details><summary>Answer</summary>

`HEAD` points to what you currently have checked out, usually a branch name, which in turn points to a commit. **Detached HEAD** means HEAD points directly at a commit (e.g. after `git checkout <sha>`). New commits there aren't on any branch and are easy to lose, so create a branch (`git switch -c name`) to keep them.
</details>

---
# 8 · SQL & databases
Reported: pros and cons of views.

**Q107. ⭐ What is a database view? Pros and cons?**

<details><summary>Answer</summary>

A **view** is a **named, stored query** that acts like a virtual table (`CREATE VIEW v AS SELECT ...`). It stores no data itself (unless it's materialised).
**Pros:** hides complex joins behind a simple name (**abstraction/reuse**), **security** (expose only certain columns or rows to certain users), a **stable interface** when underlying tables change, and consistent business logic in one place.
**Cons:** **performance can hide behind simplicity**: the query runs every time, nested views produce complex plans, and the optimiser may not push filters down well. Views are often **not updatable** (joins, aggregates). They add **dependencies** (changing a base table can break them), and debugging gets harder.
**Materialised view:** stores the result physically, so reads are fast, but the data can be **stale** and needs **refreshing** (good for dashboards and daily P&L aggregates).
</details>

**Q108. Types of JOIN?**

<details><summary>Answer</summary>

**INNER**: only matching rows. **LEFT (OUTER)**: all left rows, with NULLs where the right side has no match. **RIGHT**: the mirror image. **FULL OUTER**: all rows from both. **CROSS**: Cartesian product. **Self join**: a table joined to itself (e.g. employee → manager).
*Finance flavour:* an **as-of join** (match each trade to the latest quote at or before its time) is common in kdb+/q and pandas (`merge_asof`).
</details>

**Q109. What is an index? Tradeoffs?**

<details><summary>Answer</summary>

A separate data structure (usually a **B+ tree**, sometimes a hash) that speeds up lookups, range scans and sorting on some columns, turning a full scan O(n) into O(log n). Costs: **slower writes** (every INSERT/UPDATE must update the indexes), **extra storage**, and little benefit on low-selectivity columns. A **composite index** `(a, b)` helps queries filtering on `a`, or on `a` and `b`, but not on `b` alone (leftmost prefix). A **clustered** index determines the physical row order.
</details>

**Q110. WHERE vs HAVING? GROUP BY?**

<details><summary>Answer</summary>

`WHERE` filters **rows before** grouping; `HAVING` filters **groups after** aggregation. Example: `SELECT sym, SUM(qty) FROM trades WHERE ts >= '2026-10-09' GROUP BY sym HAVING SUM(qty) > 1000;`
</details>

**Q111. What are window functions?**

<details><summary>Answer</summary>

Aggregates computed **over a window of rows without collapsing them**: `ROW_NUMBER()`, `RANK()`, `LAG()`/`LEAD()`, running sums. Example, the previous price per symbol: `LAG(px) OVER (PARTITION BY sym ORDER BY ts)`. A 5-trade moving average: `AVG(px) OVER (PARTITION BY sym ORDER BY ts ROWS 4 PRECEDING)`.
</details>

**Q112. What is ACID?**

<details><summary>Answer</summary>

**Atomicity** (all or nothing), **Consistency** (constraints hold before and after), **Isolation** (concurrent transactions don't see each other's partial work), **Durability** (committed data survives crashes, via the write-ahead log). Isolation levels: read uncommitted < read committed < repeatable read < serializable, trading anomalies (dirty/non-repeatable/phantom reads) against concurrency.
</details>

**Q113. What is normalisation? When would you denormalise?**

<details><summary>Answer</summary>

Normalisation (1NF → 2NF → 3NF) removes redundancy so every fact is stored once, which avoids update anomalies. **Denormalise** (duplicate data) for **read performance** in analytics and reporting, where writes are rare and joins are expensive. Columnar time-series stores (kdb+, ClickHouse) are common in finance.
</details>

**Q114. Primary key vs foreign key vs unique?**

<details><summary>Answer</summary>

**Primary key:** unique, non-null row identifier (one per table). **Unique:** no duplicates (NULLs usually allowed). **Foreign key:** a column referencing another table's primary key, which enforces **referential integrity** (e.g. `orders.account_id → accounts.id`).
</details>

---
# 9 · Data structures (quick fire)

**Q115. ⭐ Hash table: how does it work, and what's the worst case?**

<details><summary>Answer</summary>

Hash the key to a bucket index. Collisions are handled by **chaining** (lists per bucket) or **open addressing** (probe for the next free slot: linear, quadratic, double hashing). O(1) average; **O(n) worst case** when everything collides (bad hash or an adversarial input). Resize when the load factor gets high. Rehashing is O(n), but amortised O(1) per insert.
</details>

**Q116. Array vs linked list?**

<details><summary>Answer</summary>

Array: O(1) random access, **contiguous (cache-friendly)**, O(n) insert in the middle. Linked list: O(1) insert/delete **given a node**, O(n) access, pointer overhead, cache misses. In practice arrays/vectors usually win, unless you need stable addresses or O(1) splicing (e.g. the order queue at a price level, the LRU list).
</details>

**Q117. Heap vs BST?**

<details><summary>Answer</summary>

**Binary heap:** O(1) min/max, O(log n) push/pop, stored compactly in an array, but **no ordered traversal** and O(n) arbitrary search. **Balanced BST** (red-black, AVL): O(log n) insert/delete/search, **ordered iteration**, successor/predecessor queries. For an order book you need ordered levels **and** deletion of arbitrary levels, so a BST (or a price-indexed array) fits, and a heap is awkward.
</details>

**Q118. Stack vs queue: applications?**

<details><summary>Answer</summary>

Stack (LIFO): function calls, undo, expression parsing, DFS, bracket matching. Queue (FIFO): BFS, task scheduling, buffering, **order matching at a price level (price-time priority)**.
</details>

**Q119. What's the difference between BFS and DFS?**

<details><summary>Answer</summary>

BFS explores level by level with a queue: it finds **shortest paths in unweighted graphs**, but memory is O(width). DFS goes deep first with a stack or recursion: memory is O(depth), and it's good for connectivity, cycle detection and topological sort. Dijkstra = BFS with a priority queue for non-negative weights.
</details>

**Q120. What is amortised complexity?**

<details><summary>Answer</summary>

The average cost per operation over a **worst-case sequence** of operations. Example: vector `push_back` is occasionally O(n) (on reallocation), but doubling the capacity means the total cost of n pushes is O(n), so it's **O(1) amortised**.
</details>

---
# 10 · ML trivia (light: you have the AI minor)

**Q121. ⭐ Bias–variance tradeoff?**

<details><summary>Answer</summary>

Error = bias² + variance + noise. **High bias:** the model is too simple, so it **underfits** (bad on train and test). **High variance:** the model is too complex or fits the noise, so it **overfits** (great on train, bad on test). Balance it with model complexity, regularisation and more data. *Finance:* noisy data with a low signal-to-noise ratio means overfitting is the main danger.
</details>

**Q122. ⭐ How do you prevent overfitting?**

<details><summary>Answer</summary>

More or better data, **regularisation** (L1 gives sparsity, L2 shrinks weights), dropout, early stopping, simpler models, feature selection, and cross-validation. *In finance:* use **walk-forward / time-series cross-validation** (never shuffle across time), avoid **look-ahead bias** and **survivorship bias**, and beware of multiple-testing / data snooping.
</details>

**Q123. L1 vs L2 regularisation?**

<details><summary>Answer</summary>

L1 (Lasso) adds λΣ|w|: it pushes weights **exactly to zero** (feature selection). L2 (Ridge) adds λΣw²: it **shrinks** all weights smoothly, and handles correlated features better. Elastic Net combines both.
</details>

**Q124. Precision vs recall?**

<details><summary>Answer</summary>

Precision = TP / (TP + FP): of the things predicted positive, how many really are. Recall = TP / (TP + FN): of the actual positives, how many were caught. The tradeoff is controlled by the decision threshold. F1 is their harmonic mean.
</details>

**Q125. Why is train/test splitting different for time series?**

<details><summary>Answer</summary>

Random shuffling **leaks the future into training**. Use chronological splits (train on the past, test on the future), walk-forward validation, and add purging/embargo gaps so overlapping labels don't leak.
</details>

**Q126. Gradient descent: batch vs stochastic vs mini-batch?**

<details><summary>Answer</summary>

Batch computes the gradient over the whole dataset (stable, slow). SGD uses one sample (noisy, fast, can escape shallow minima). Mini-batch is the standard compromise (it vectorises well on GPUs). Optimisers such as momentum and Adam adapt the step size per parameter.
</details>

---
# 11 · Finance & company context

**Q127. What does Squarepoint do?**

<details><summary>Answer</summary>

**Squarepoint Capital** is a global **quantitative investment management** firm (a hedge fund) running **systematic** strategies across asset classes (equities, futures, FX, fixed income, commodities). It began as **nQuant**, a quant trading group inside **Lehman Brothers** that moved to **Barclays Capital** after Lehman collapsed in 2008, then **spun out as Squarepoint in 2014**. Large offices in London, New York, Singapore and more. Technology-heavy: researchers, developers and infrastructure all work closely together.
*Check their website before the interview* for current facts (AUM and headcount change over time).
</details>

**Q128. ⭐ Systematic vs discretionary trading?**

<details><summary>Answer</summary>

**Systematic:** decisions come from **rules/models** (statistical signals, ML) that are researched, backtested and executed automatically. It's scalable, consistent and unemotional, but it depends on data quality and is vulnerable to regime changes and overfitting. **Discretionary:** a human PM makes the calls, using judgement, macro views and fundamental analysis (possibly supported by data). Squarepoint is primarily **systematic**, so good software, data pipelines and research infrastructure *are* the edge.
</details>

**Q129. What does a SWE do at a systematic fund?**

<details><summary>Answer</summary>

Builds and maintains **research platforms** (backtesting, data access at scale), **data pipelines** (ingesting and cleaning vendor and market data, which is why "clean JSON" and "regex over raw data" show up in interviews), **execution and trading systems** (order management, exchange connectivity, low-latency C++), **risk and monitoring** tools, and infrastructure (distributed compute, storage). Python for research and tooling; C++ for performance-critical parts.
</details>

**Q130. ⭐ What is a limit order book? Market vs limit order?**

<details><summary>Answer</summary>

The LOB is the list of all resting **limit orders** at each price: **bids** (buy) and **asks** (sell). **Best bid/ask** = top of book; **spread** = ask − bid. A **limit order** specifies a price, so it rests in the book if it isn't immediately marketable (it **provides liquidity**). A **market order** executes immediately against the best available prices (it **takes liquidity**, and pays the spread and slippage). Matching is **price-time priority**: best price first, then first-in-first-out at the same price.
</details>

**Q131. What is market making? What is alpha? What is slippage?**

<details><summary>Answer</summary>

**Market making:** continuously quoting both a bid and an ask, earning the spread while managing **inventory risk** and **adverse selection** (trading against better-informed flow). **Alpha:** return in excess of what's explained by market or factor exposure, i.e. the model's real edge. **Slippage / implementation shortfall:** the difference between the decision price and the actual execution price, which is what execution algorithms (and your RL execution agent) try to minimise. **VWAP/TWAP:** benchmark execution algorithms that slice an order over time.
</details>

**Q132. Why does latency matter even for a non-HFT fund?**

<details><summary>Answer</summary>

Even medium-frequency strategies lose alpha to **slippage** if execution is slow; stale data leads to bad decisions; and risk checks must be fast. For research, "latency" means **throughput**: how fast you can backtest over years of tick data.
</details>

---
# 12 · Your résumé: defend every line
Opening minutes: résumé deep-dive. Write your own answers in the empty cells; reading them the night before is the best prep.

### Limit Order Book reconstructor → execution simulator + RL agent (your strongest story)
Prepare **30-second** and **2-minute** versions: *problem → your design → hardest bug → result/number → what you'd improve*.

Likely questions:
1. What is ITCH 5.0? How are messages framed? Which message types did you handle (Add, Add-with-MPID, Execute, Execute-with-price, Cancel, Delete, Replace, Trade...)?
2. **Which data structures hold the book, and why?** (price levels sorted: tree or array by tick; order-id → order hash map; FIFO per level). What's the complexity of add / cancel / execute / best bid-ask?
3. How do you handle **Order Replace** (new order id, loses time priority) and partial executions?
4. How did you validate correctness? (e.g. compare reconstructed top-of-book or trade prints against a reference, invariants such as best bid < best ask, no negative quantities)
5. Throughput: how many messages per second? Where was the bottleneck? (parsing? Python objects? allocation?) Why Rust for that part?
6. Why is ITCH big-endian binary, and how do you parse it efficiently (`struct.unpack`, memory-mapping, zero-copy)?
7. How would you **rewrite it in C++** for production? (pre-allocated order pool, intrusive lists, flat arrays by price tick, no allocation on the hot path, SPSC queue from the feed thread)
8. Simulator realism: how do you model **queue position**, **latency**, **market impact**, and fills for your own orders? What's unrealistic about it?
9. RL agent: state / action / reward? Baseline (TWAP/VWAP)? How do you avoid overfitting and look-ahead in training? How do you evaluate it?

In [ ]:
# My 30-second LOB pitch:
"""

"""

In [ ]:
# Numbers to remember (msgs/sec, dataset size, # message types, latency per message, RL result vs baseline):
"""

"""

### Your C++ project
For **whichever C++ project is on your CV**, prepare answers to these. Interviewers use them to check whether you really wrote C++:
1. Walk me through the architecture. What are the main classes and who **owns** what?
2. Where did you use **smart pointers**? Why `unique_ptr` vs `shared_ptr` there? Any raw pointers, and why are they safe?
3. Which STL containers did you choose and why? Any `reserve`, `emplace_back`, iterator-invalidation issues?
4. Did you implement copy/move constructors? Rule of 0/3/5 in your classes?
5. Any **polymorphism** (virtual functions)? Did the base class have a virtual destructor?
6. Did you use threads? How did you synchronise? Any race or deadlock you had to debug?
7. Hardest bug (segfault? leak? UB?), and **how you found it** (gdb, ASan, Valgrind, printf)?
8. Build system (CMake?), compiler flags, tests (GoogleTest?).
9. How would you measure and improve its performance?
10. What would you change if you started over?

In [ ]:
# My C++ project: architecture in 3 sentences, ownership model, hardest bug + how I found it:
"""

"""

### Other projects: one-liners to have ready
- **Grounded-RAG via RL** (DPO/GRPO): what's the difference between DPO and PPO-style RLHF? How did you measure groundedness?
- **Current TD internship**: what you built, the stack, one measurable impact (you'll have just demoed it on Oct 6, so reuse that!)
- Behavioural: **Why Squarepoint? Why systematic trading? Why SWE and not QR?** A time you debugged something hard. A time you disagreed with a teammate.

In [ ]:
# Why Squarepoint (3 sentences: systematic + tech-driven, my LOB/RL work, what I want to learn):
"""

"""

---
# 13 · Questions to ask them (last 2–3 minutes)

Pick 2:
- What does a typical intern project look like, and how does it get to production?
- How do developers and quant researchers work together day to day? Is it embedded or a platform team?
- What's the split between Python and C++ in the team this role sits in?
- How do you test and validate changes to trading or research infrastructure before they go live?
- What distinguishes interns who get return offers?
- What's a recent technical challenge the team is excited about?

---
### Night-before checklist (Oct 8)
- [ ] Re-read every ⭐ answer (about 45 min)
- [ ] Say the TCP vs UDP + multicast answer out loud once
- [ ] Say the smart-pointer, dangling-pointer and virtual-destructor answers out loud once
- [ ] Read your LOB pitch + C++ project notes
- [ ] GIL / threading vs multiprocessing / generators / decorators / context managers: 1 sentence each
- [ ] Sleep 😴